<a href="https://colab.research.google.com/github/K-Sanyam-130/AI_LAB/blob/main/Forward_Chaining_FOL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class FOLForwardChaining:
    def __init__(self, initial_kb, rules):
        # Work with copies to avoid mutating shared global state unexpectedly
        self.kb = list(initial_kb)
        self.rules = rules

    def unify(self, pattern, fact, subst):
        if len(pattern) != len(fact):
            return None
        for p, f in zip(pattern, fact):
            if p.startswith("?"):
                if p in subst:
                    if subst[p] != f:
                        return None
                else:
                    subst[p] = f
            elif p != f:
                return None
        return subst

    def match_all(self, conditions, facts, subst=None):
        if subst is None:
            subst = {}
        if not conditions:
            return [subst]
        results = []
        first = conditions[0]
        remaining = conditions[1:]
        for fact in facts:
            new_subst = self.unify(first, fact, subst.copy())
            if new_subst is not None:
                results.extend(self.match_all(remaining, facts, new_subst))
        return results

    def apply_substitution(self, expression, subst):
        return tuple(
            subst.get(item, item) if item.startswith("?") else item
            for item in expression
        )

    def run(self, goal=None, verbose=True):
        derived = True
        if verbose:
            print("--- Starting Forward Chaining Inference ---")
        while derived:
            derived = False
            for conditions, conclusion in self.rules:
                substitutions = self.match_all(conditions, self.kb)
                for subst in substitutions:
                    new_fact = self.apply_substitution(conclusion, subst)
                    if new_fact not in self.kb:
                        self.kb.append(new_fact)
                        if verbose:
                            print(f"Derived New Fact: {new_fact}")
                        derived = True

        if verbose:
            print("--- Inference Complete ---\n")

        if goal is not None:
            success = goal in self.kb
            if verbose:
                if success:
                    print(f"SUCCESS: Goal {goal} is proven!")
                else:
                    print(f"FAILURE: Goal {goal} could not be proven.")
            return success, self.kb
        return self.kb
import ast

def parse_tuple_input(prompt_text):
    """Helper to parse a user-inputted tuple or list of tuples."""
    val = input(prompt_text).strip()
    if not val:
        return []
    try:
        # Safely evaluate the literal structure entered by the user
        parsed = ast.literal_eval(val)
        return parsed
    except Exception as e:
        print(f"Error parsing input. Please ensure it is in valid Python format. Error: {e}")
        return None

def interactive_solver():
    print("=== Interactive FOL Forward Chaining Solver ===\n")

    # 1. Get Initial KB Facts
    print("Step 1: Enter your Initial Knowledge Base (Facts).")
    print("Example format: [(\"American\", \"Robert\"), (\"Missile\", \"T1\")]")
    kb_input = None
    while kb_input is None:
        kb_input = parse_tuple_input("Enter KB Facts: ")

    # 2. Get Rules
    print("\nStep 2: Enter your Inference Rules.")
    print("Example format: [ ([(\"Missile\", \"?x\")], (\"Weapon\", \"?x\")) ]")
    rules_input = None
    while rules_input is None:
        rules_input = parse_tuple_input("Enter Inference Rules: ")

    # 3. Get Target Goal
    print("\nStep 3: Enter your Target Goal.")
    print("Example format: (\"Criminal\", \"Robert\")")
    goal_input = None
    while goal_input is None:
        goal_input = parse_tuple_input("Enter Target Goal: ")
        if isinstance(goal_input, list) and len(goal_input) == 1:
            goal_input = goal_input[0]

    # 4. Instantiate and Run
    print("\nProcessing your input and running inference...")
    engine = FOLForwardChaining(initial_kb=kb_input, rules=rules_input)
    engine.run(goal=goal_input, verbose=True)

# Run the interactive interface
interactive_solver()

In [2]:
# Define the inputs programmatically to avoid any parsing/input formatting issues
kb_facts = [
    ("American", "Robert"),
    ("Missile", "T1"),
    ("Owns", "Robert", "T1"),
    ("Sells", "Robert", "T1", "A"),
    ("Enemy", "America", "A")
]

# Rules should be a list of tuples: [ (conditions, conclusion) ]
inference_rules = [
    ([("Missile", "?x")], ("Weapon", "?x")),
    ([("Enemy", "?c", "?s"), ("American", "?p"), ("Sells", "?p", "?w", "?s"), ("Weapon", "?w")], ("Criminal", "?p"))
]

target_goal = ("Criminal", "Robert")

# Instantiate and run the engine
print("--- Running Forward Chaining Inference --- ")
engine = FOLForwardChaining(initial_kb=kb_facts, rules=inference_rules)
success, final_kb = engine.run(goal=target_goal, verbose=True)

--- Running Forward Chaining Inference --- 
--- Starting Forward Chaining Inference ---
Derived New Fact: ('Weapon', 'T1')
Derived New Fact: ('Criminal', 'Robert')
--- Inference Complete ---

SUCCESS: Goal ('Criminal', 'Robert') is proven!


In [4]:
def run_interactive_solver():
    print("=== FOL Forward Chaining Solver (User Input Mode) ===\n")
    print("Please provide the input data in valid Python syntax (lists and tuples).\n")

    # 1. Facts Input
    print("Step 1: Enter Initial Facts (List of tuples)")
    print("Example: [(\"American\", \"Robert\"), (\"Missile\", \"T1\")]")
    kb_input = None
    while kb_input is None:
        kb_input = parse_tuple_input("Enter KB Facts: ")

    # 2. Rules Input
    print("\nStep 2: Enter Inference Rules (List of tuples of (conditions, conclusion))")
    print("Example: [ ([(\"Missile\", \"?x\")], (\"Weapon\", \"?x\")) ]")
    rules_input = None
    while rules_input is None:
        rules_input = parse_tuple_input("Enter Inference Rules: ")
        # Standardize single-rule input if the user forgot the outer list bracket
        if isinstance(rules_input, tuple) and len(rules_input) == 2 and isinstance(rules_input[0], list):
            rules_input = [rules_input]

    # 3. Goal Input
    print("\nStep 3: Enter Target Goal (Single tuple)")
    print("Example: (\"Criminal\", \"Robert\")")
    goal_input = None
    while goal_input is None:
        goal_input = parse_tuple_input("Enter Target Goal: ")
        if isinstance(goal_input, list) and len(goal_input) == 1:
            goal_input = goal_input[0]

    # 4. Run solver
    print("\n--- Starting Inference Engine ---")
    engine = FOLForwardChaining(initial_kb=kb_input, rules=rules_input)
    engine.run(goal=goal_input, verbose=True)

run_interactive_solver()

=== FOL Forward Chaining Solver (User Input Mode) ===

Please provide the input data in valid Python syntax (lists and tuples).

Step 1: Enter Initial Facts (List of tuples)
Example: [("American", "Robert"), ("Missile", "T1")]
Enter KB Facts: [("American", "Robert"), ("Missile", "T1"), ("Owns", "Robert", "T1"), ("Sells", "Robert", "T1", "A"), ("Enemy", "America", "A")]

Step 2: Enter Inference Rules (List of tuples of (conditions, conclusion))
Example: [ ([("Missile", "?x")], ("Weapon", "?x")) ]
Enter Inference Rules: [([("Missile", "?x")], ("Weapon", "?x")), ([("Enemy", "?c", "?s"), ("American", "?p"), ("Sells", "?p", "?w", "?s"), ("Weapon", "?w")], ("Criminal", "?p"))]

Step 3: Enter Target Goal (Single tuple)
Example: ("Criminal", "Robert")
Enter Target Goal: ("Criminal", "Robert")

--- Starting Inference Engine ---
--- Starting Forward Chaining Inference ---
Derived New Fact: ('Weapon', 'T1')
Derived New Fact: ('Criminal', 'Robert')
--- Inference Complete ---

SUCCESS: Goal ('Crimi